# Josh Test Notebook

Reading, processing and building a genie Agent on top of a Simulation Report document-'FEM-30044_Report.pdf' to be able to Answer Engieering Questions from the Users


| Step | What it does | AI function |
| --- | --- | --- |
| 1 | Load the PDF | none |
| 2 | Parse it into text, tables and figures | `ai_parse_document` |
| 3 | One row per element (for full-text questions) | none |
| 4 | Extract the report data into a schema built for this report | `ai_extract` |
| 5 | Write an engineering summary | `ai_query` |
| 6 | Build typed tables for Genie | none |
| 7 | Read the chart data points (force and stress-strain curves) | none |
| 8 | Read the 3D result plots from the page images | `ai_query` with a vision model |
| 9 | Measure the colour bands of the result plots | none (OpenCV) |
| 10 | Add table and column descriptions | none |
| 11 | Build the Genie agent | none |



In [0]:
# OpenCV for the colour-band step (Step 9). Version 4.11.0.86 is pinned on purpose: OpenCV 5.x bundles an OpenSSL
%pip install -q opencv-python-headless==4.11.0.86
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Configuration: the one report this notebook processes, where the tables go, and the Genie agent settings
import json

from pyspark.sql import Window
from pyspark.sql import functions as F 

CATALOG = "tedatacatalog_sandbox"
SCHEMA = "te672648"
SOURCE_FILE = "/Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf"
IMAGE_OUTPUT_PATH = "/Volumes/tedatacatalog_sandbox/te672648/josh-vol/page_images/"  # page images, outside drawings/

LLM_ENDPOINT = "databricks-claude-sonnet-4-6"  # writes the report summary
VISION_ENDPOINT = "databricks-claude-sonnet-4-6"  # reads the result-plot page images; must accept image input
REPARSE = False    # True = parse the PDF again even if it has not changed
REEXTRACT = False  # True = re-run ai_extract and ai_query (e.g. after editing the schema or prompt)

# Genie agent (Step 11, created in the Genie UI)---pending access to SQL Warehouse
GENIE_TITLE = "CAE Report 30044 Assistant"
GENIE_DESCRIPTION = ("Ask questions about TE CAE report 30044 (structural analysis of the Deutz-Leavy seal wire "
                     "distributor cover): materials, load cases, insertion and retention forces, strain and deformation results.")


def tbl(name: str) -> str:
    return f"{CATALOG}.{SCHEMA}.cae_{name}"


# Working tables
T_PARSED = tbl("parsed_report")        # ai_parse_document output (VARIANT)
T_EXTRACTED = tbl("report_extracted")  # ai_extract output (VARIANT)
T_SUMMARY = tbl("report_summary")      # ai_query summary
# Genie tables (typed, one subject each)
T_REPORT = tbl("report")               # 1 row: header, CAD and model info, conclusion, AI summary
T_ELEMENTS = tbl("report_elements")    # every text block, table and figure, for full-text questions
T_MODEL = tbl("model_data")            # nodes and elements per analysis case
T_MATERIALS = tbl("materials")         # material data tables
T_LOAD_CASES = tbl("load_cases")       # loads and boundary conditions
T_RESULTS = tbl("results_summary")     # summary-of-results table (forces, safe/fail)
T_PLOTS = tbl("result_plots")          # strain and deformation plots (max/min per material)
T_CURVES = tbl("curves")               # data points of the force-displacement and stress-strain charts
T_PLOT_VISION = tbl("result_plot_visuals")  # what the result plots show, read from the page images by a vision model
T_PLOT_BANDS = tbl("result_plot_colour_bands")  # OpenCV: share of each legend colour band per result plot
T_REMARKS = tbl("remarks")             # remarks / assumptions
GENIE_TABLES = [T_REPORT, T_MATERIALS, T_LOAD_CASES, T_RESULTS, T_PLOTS, T_PLOT_VISION, T_PLOT_BANDS, T_CURVES,
                T_MODEL, T_REMARKS, T_ELEMENTS]


def sql_str(value: str) -> str:
    """Quote a Python string as a SQL string literal for use inside F.expr()."""
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"


def save(df, table: str) -> None:
    """Write a DataFrame as a Delta table, replacing the previous version."""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: {spark.table(table).count()} rows")


print(f"Source: {SOURCE_FILE}\nTarget: {CATALOG}.{SCHEMA}.cae_*")

Source: /Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf
Target: tedatacatalog_sandbox.te672648.cae_*


## Step 1: Load the report

Reads `FEM-30044_Report.pdf` as raw bytes with Spark's `binaryFile` source. The result is one row with the path, size, modification time and content. The modification time tells later steps whether the PDF has changed since it was last parsed.

In [0]:
file_df = (
    spark.read.format("binaryFile")
    .load(SOURCE_FILE)
    .select(
        "path",
        F.element_at(F.split("path", "/"), -1).alias("file_name"),
        F.col("length").alias("file_size_bytes"),
        F.col("modificationTime").alias("file_modified_at"),
        "content",
    )
)
assert file_df.count() == 1, f"Expected exactly one file at {SOURCE_FILE}"
display(file_df.drop("content"))

path,file_name,file_size_bytes,file_modified_at
dbfs:/Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf,FEM-30044_Report.pdf,890031,2026-11-03T17:19:13.000Z


## Step 2: Parse the PDF with `ai_parse_document`

Turns the 8-page report into a structured VARIANT. It contains every section header, text block, table (stored as HTML), caption and figure. With `descriptionElementTypes '*'`, the AI also describes each figure, and it turns charts such as the force-displacement curves into JSON data. Page images are saved to `page_images/`.

Parsing is the slowest and most expensive step, so it is skipped when the PDF has not changed since the last run. Set `REPARSE = True` to force it.

In [0]:
PARSE_EXPR = (
    "ai_parse_document(content, map('version', '2.0', 'descriptionElementTypes', '*', "
    f"'imageOutputPath', {sql_str(IMAGE_OUTPUT_PATH)}))"
)

unchanged = (
    not REPARSE
    and spark.catalog.tableExists(T_PARSED)
    and spark.table(T_PARSED).where("parse_ok")
        .join(file_df.select("path", "file_modified_at"), ["path", "file_modified_at"]).count() > 0
)

if unchanged:
    print("PDF unchanged since the last parse; reusing", T_PARSED)
else:
    parsed = (
        file_df
        .withColumn("parsed", F.expr(PARSE_EXPR))
        .drop("content")
        .withColumn("parse_ok", F.expr("parsed IS NOT NULL AND is_variant_null(parsed:error_status)"))
        .withColumn("parse_error", F.expr(
            "CASE WHEN parsed IS NULL THEN 'no parse output' "
            "WHEN is_variant_null(parsed:error_status) THEN NULL "
            "ELSE cast(parsed:error_status AS STRING) END"))
        .withColumn("page_count", F.expr("size(try_cast(parsed:document:pages AS ARRAY<VARIANT>))"))
        .withColumn("parsed_at", F.current_timestamp())
    )
    save(parsed, T_PARSED)

parsed_df = spark.table(T_PARSED).where("parse_ok").select("path", "file_name", "parsed", "parsed_at")
display(spark.table(T_PARSED).select("file_name", "page_count", "parse_ok", "parse_error", "parsed_at"))

PDF unchanged since the last parse; reusing tedatacatalog_sandbox.te672648.cae_parsed_report


file_name,page_count,parse_ok,parse_error,parsed_at
FEM-30044_Report.pdf,8,true,null,2026-11-04T17:08:00.576Z


## Step 3: One row per report element

Splits the parsed VARIANT into one row per element: section header, text, table, caption, figure, page header or footer. Each row keeps its page number (1-8), its content (tables as HTML) and the AI's figure description. Genie uses this table for questions about anything the structured tables don't cover, such as "what does the report say about glass fibre?". No AI is used here.

In [0]:
elements = (
    parsed_df
    .selectExpr("path", "file_name", "posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (element_index, el)")
    .selectExpr(
        "path", "file_name", "element_index",
        "el:type::STRING AS element_type",
        "try_cast(el:bbox[0]:page_id AS INT) + 1 AS page_number",  # page_id is 0-based
        "el:content::STRING AS content",
        "el:description::STRING AS figure_description",
    )
)
save(elements, T_ELEMENTS)
display(spark.table(T_ELEMENTS).groupBy("page_number").pivot("element_type").count().orderBy("page_number"))

tedatacatalog_sandbox.te672648.cae_report_elements: 120 rows


page_number,caption,figure,page_footer,page_header,page_number,section_header,table,text,title
1,null,2,1,null,null,3,4,3,1
2,null,3,2,2,1,4,4,4,null
3,2,6,2,2,1,3,2,5,null
4,2,3,2,2,1,2,1,3,null
5,2,3,2,2,1,3,null,null,null
6,2,3,2,2,1,1,null,null,null
7,2,3,2,2,1,2,null,null,null
8,2,3,2,2,1,1,null,null,null


## Step 4: Extract the report data with `ai_extract`

One `ai_extract` call reads the whole report and fills in a schema that follows its layout:

| Report section | Fields |
| --- | --- |
| Header table and footer | report number, project number, date, description, analysis title and type, customer, analyst, requested by, released by, distribution |
| 1. Objective | objective |
| 3. Summary of results | process, cover material, force per single latch and per full model (N), cover and housing assessment |
| 4.1-4.2 CAD and model data | CAD part number, file and date; software, element types, nodes and elements per case |
| 4.3 Material data | designation, description, ID, material law, Young's modulus, tensile strength, elongation (each value and the value in parentheses marked *), Poisson's ratio, cover or housing |
| 4.4 Remarks | remarks, contact friction, model simplification |
| 4.6 Boundary conditions | process, load step, description |
| 5. Results plots | figure, process, condition, result type, part, material, unit, max and min, observation |

Numbers are extracted as numbers without units, so Genie can compare and sort them. The result is cached and only extracted again when the PDF is re-parsed or `REEXTRACT = True`.

In [0]:
def s(desc): return {"type": "string", "description": desc}
def n(desc): return {"type": "number", "description": desc}
def i(desc): return {"type": "integer", "description": desc}
def strings(desc): return {"type": "array", "description": desc, "items": {"type": "string"}}
def objects(desc, props): return {"type": "array", "description": desc, "items": {"type": "object", "properties": props}}

EXTRACTION_SCHEMA = {
    # Header table, page header and footer
    "report_number": s("CAE Report No., e.g. 30044"),
    "cae_team": s("CAE team in the page header, e.g. CAE EMEA"),
    "project_number": s("Project No. from the header table"),
    "report_date": s("Date from the header table, in YYYY-MM-DD format"),
    "part_description": s("Description from the header table"),
    "analysis_title": s("Analyses title from the header table"),
    "analysis_type": s("Analyses type from the header table, e.g. Mechanical"),
    "customer": s("Customer from the header table"),
    "analyst": s("CAE Analyst"),
    "requested_by": s("Requested by"),
    "released_by": s("Released by"),
    "distribution": strings("People on the distribution list, names only"),
    # 1. Objective and overall conclusion
    "objective": s("Text of section 1, Objective of Analyses"),
    "conclusion": s("Overall conclusion of the report, from the summary of results and the observations on the results pages"),
    # 4.1-4.2 CAD and model data
    "cad_part_number": s("Part No. from the CAD data table"),
    "cad_file_name": s("File Name from the CAD data table"),
    "cad_date": s("Date from the CAD data table, in YYYY-MM-DD format"),
    "software": s("Software / Release from the CAE model data table"),
    "element_types": s("Type of Elements from the CAE model data table"),
    "model_data": objects("CAE model data table, one item per analysis case", {
        "case": s("Insertion or Retention"),
        "number_of_nodes": i("Number of Nodes"),
        "number_of_elements": i("Number of Elements")}),
    # 4.3 Material data
    "materials": objects("One item per material data table (Material For Cover / Material For Housing)", {
        "designation": s("Material Designation, e.g. Lanxess Pocan B3215"),
        "description": s("Material Description, e.g. PBT GF10"),
        "material_id": s("Material ID"),
        "material_law": s("Material Law, e.g. MISO"),
        "used_for": s("Part the material is used for, from the table caption: Cover or Housing"),
        "youngs_modulus_mpa": n("Young's Modulus [MPa], the first value"),
        "youngs_modulus_ref_mpa": n("Young's Modulus [MPa], the value in parentheses marked *"),
        "tensile_strength_mpa": n("Tensile Strength [MPa], the first value"),
        "tensile_strength_ref_mpa": n("Tensile Strength [MPa], the value in parentheses marked *"),
        "elongation_pct": n("Elongation [%], the first value"),
        "elongation_ref_pct": n("Elongation [%], the value in parentheses marked *"),
        "poissons_ratio": n("Poisson's Ratio")}),
    # 4.4 Remarks
    "remarks": strings("Bullet points in section 4.4 Remarks"),
    "contact_friction": n("Contact friction coefficient used in the FE simulation"),
    "model_simplification": s("Model simplification stated in the remarks, e.g. quarter model"),
    # 4.6 Boundary conditions
    "load_cases": objects("Loads and boundary conditions (section 4.6), one item per applied load. Do not add "
                          "items for headings or labels.", {
        "process": s("Insertion or Retention. The Insertion process has a single load (remote displacement along "
                     "-ve Y). The load steps labelled LS1, LS2 and LS3 (deflect the latch, release it, pull the "
                     "cover along +ve Y) all belong to the Retention process"),
        "load_step": s("Load step label, e.g. LS1; null if none"),
        "description": s("Load or boundary condition as printed")}),
    # 3. Summary of results
    "results_summary": objects("Rows of the section 3 Summary of Results table", {
        "process": s("Insertion or Retention"),
        "material": s("Material of Cover"),
        "force_text": s("Force cell exactly as printed"),
        "force_single_latch_n": n("Force for a single latch, in N"),
        "force_full_model_n": n("Force for the full model, in N"),
        "cover_assessment": s("Cover column as printed, e.g. Safe"),
        "housing_assessment": s("Housing column as printed, e.g. Safe")}),
    # 5. Results plots
    "result_plots": objects("Section 5 strain and deformation plots: one item per material in each figure", {
        "figure_number": i("Figure number from the caption"),
        "process": s("Insertion or Retention"),
        "condition": s("Maximum deflection condition or Failure condition, from the caption"),
        "result_type": s("Equivalent total strain or Directional deformation (Y axis)"),
        "part": s("Cover or Housing, from the caption; null if not stated"),
        "material": s("Material named on the plot"),
        "unit": s("Unit, e.g. mm/mm or mm"),
        "max_value": n("Max value on the legend"),
        "min_value": n("Min value on the legend"),
        "observation": s("Observation text printed with the plot, if any")}),
}
EXTRACTION_INSTRUCTIONS = (
    "This EA simulation report. Read every table, caption and plot legend. "
    "Copy text values as printed. For number fields give the number only, without units. "
    "Leave a field null when it is not printed in the report; never guess."
)
EXTRACT_EXPR = (f"ai_extract(parsed, {sql_str(json.dumps(EXTRACTION_SCHEMA))}, "
                f"map('version', '2.0', 'instructions', {sql_str(EXTRACTION_INSTRUCTIONS)}))")


def needs_refresh(table: str, force: bool) -> bool:
    """True if the table is missing, a re-run is forced, or it was built from an older parse."""
    if force or not spark.catalog.tableExists(table):
        return True
    return spark.table(table).join(parsed_df.select("path", "parsed_at"), ["path", "parsed_at"]).count() == 0


if needs_refresh(T_EXTRACTED, REEXTRACT):
    extracted = (
        parsed_df
        .withColumn("extracted", F.expr(EXTRACT_EXPR))
        .drop("parsed")
        .withColumn("extract_error", F.expr("try_cast(extracted:error_message AS STRING)"))
        .withColumn("extracted_at", F.current_timestamp())
    )
    save(extracted, T_EXTRACTED)
else:
    print("Extraction is up to date; reusing", T_EXTRACTED)

extracted_df = spark.table(T_EXTRACTED).where("extract_error IS NULL")
display(spark.table(T_EXTRACTED).select("file_name", "extract_error", "extracted"))

Extraction is up to date; reusing tedatacatalog_sandbox.te672648.cae_report_extracted


file_name extract_error extracted FEM-30044_Report.pdf null {"error_message":null,"metadata":{"version":"2.0"},"response":{"analysis_title":"Structural analysis of cover","analysis_type":"Mechanical","analyst":"D V, Vinay","cad_date":"2022-10-13","cad_file_name":"B-2423482-001.stp","cad_part_number":"B-2423482-001","cae_team":"CAE EMEA","conclusion":"Insertion forces are safe for both cover and housing materials. However, during retention, the cover experiences failure for both materials (Lanxess Pocan B3215 and Ultradur B4520). The housing remains safe for Lanxess Pocan B3215 but is expected to show local plastic deformation for Ultradur B4520 during retention.","contact_friction":0.2,"customer":"Deutz","distribution":["1. B, Prabu","2. A G, Deepak","3. Kumar S N, Mahesh"],"element_types":"Solid187/Target170/Contact174","load_cases":[{"description":"Remote displacement of 9 mm is applied on rigid body along -ve Y direction and fixed in all other direction.","load_step":null,"process":"Insertion"},{"description":"Latch is deflected 2 mm in -ve X direction and fixed in all other directions.","load_step":"LS1","process":"Retention"},{"description":"Applied defelection is released.","load_step":"LS2","process":"Retention"},{"description":"Remote displacement of 9 mm is applied on rigid body along +ve Y direction and fixed in all other direction.","load_step":"LS3","process":"Retention"}],"materials":[{"description":"PBT GF10","designation":"Lanxess Pocan B3215","elongation_pct":3.92,"elongation_ref_pct":4,"material_id":"767","material_law":"MISO","poissons_ratio":0.35,"tensile_strength_mpa":73,"tensile_strength_ref_mpa":72,"used_for":"Cover","youngs_modulus_mpa":3973,"youngs_modulus_ref_mpa":3954},{"description":"PBT","designation":"Ultradur B4520","elongation_pct":40.55,"elongation_ref_pct":50,"material_id":"63","material_law":"MISO","poissons_ratio":0.35,"tensile_strength_mpa":62,"tensile_strength_ref_mpa":60,"used_for":"Cover","youngs_modulus_mpa":2722,"youngs_modulus_ref_mpa":2710},{"description":"PBT GF20","designation":"Ultradur B4300G4","elongation_pct":3.44,"elongation_ref_pct":3.5,"material_id":"58","material_law":"MISO","poissons_ratio":0.35,"tensile_strength_mpa":93,"tensile_strength_ref_mpa":92,"used_for":"Housing","youngs_modulus_mpa":5412,"youngs_modulus_ref_mpa":5382}],"model_data":[{"case":"Insertion","number_of_elements":98152,"number_of_nodes":153343},{"case":"Retention","number_of_elements":98198,"number_of_nodes":154419}],"model_simplification":"Quarter model","objective":"To evaluate the Insertion and Retention forces of Cover from Housing for different materials.","part_description":"Deutz-Leavy seal Wire distributor","project_number":"22-000903154","released_by":"A G, Deepak","remarks":["Contact friction value of 0.2 has been used in FE simulation.","The data obtained is standard tensile test data, but latch part is subjected to bending, buckling, and shearing load conditions in addition to tensile behaviour, this may add to change in results among LAB test and FEA results.","80% of the stress considered to include the glass fibre effect","Quarter model is considered for simulation."],"report_date":"2022-12-20","report_number":"30044","requested_by":"Kumar H, Praveen","result_plots":[{"condition":"Maximum deflection condition","figure_number":5,"material":"Lanxess Pocan B3215","max_value":0.041553,"min_value":1.4753E-5,"observation":"Strain induced in the cover is more than the breaking stain(3.92%) of the material, but, it is in compressive region, hence, cover is safe during insertion. Strain induced in the cover is less than the breaking stain(40.55%) of the material,hence, cover is safe during insertion.","part":"Cover","process":"Insertion","result_type":"Equivalent Total Strain","unit":"mm/mm"},{"condition":"Maximum deflection condition","figure_number":5,"material":"Ultradur B4520","max_value":0.042448,"min_value":1.1486E-5,"observation":"Strain induced in the cover is less than the breaking stain(40

## Step 5: Write a summary with `ai_query`

Joins all the parsed text, tables and figure descriptions into one prompt. The model then writes a short engineering summary: what was analysed and why, the materials compared, the main forces and results, and the conclusion. Genie uses it for broad questions like "what is this report about?". The summary is cached like the extraction.

In [0]:
SUMMARY_PROMPT = (
    "You are a CAE engineer. Summarize this TE Connectivity CAE report in 5-8 sentences of plain text (no "
    "markdown): the part and customer, the objective, the analysis setup (software, model, load cases), the "
    "materials compared, the key insertion and retention forces, which parts are safe or fail, and the "
    "conclusion. Use only facts and numbers that appear in the report.\n\nREPORT CONTENT:\n"
)
FULL_TEXT_EXPR = (
    "concat_ws('\\n\\n', transform(try_cast(parsed:document:elements AS ARRAY<VARIANT>), "
    "e -> concat_ws(' ', try_cast(e:content AS STRING), try_cast(e:description AS STRING))))"
)
SUMMARY_EXPR = (f"ai_query({sql_str(LLM_ENDPOINT)}, concat({sql_str(SUMMARY_PROMPT)}, full_text), "
                "failOnError => false)")

if needs_refresh(T_SUMMARY, REEXTRACT):
    summary = (
        parsed_df
        .withColumn("full_text", F.expr(FULL_TEXT_EXPR))
        .withColumn("llm", F.expr(SUMMARY_EXPR))
        .select("path", "parsed_at",
                F.col("llm.result").alias("ai_summary"),
                F.col("llm.errorMessage").alias("summary_error"),
                F.current_timestamp().alias("summarized_at"))
    )
    save(summary, T_SUMMARY)
else:
    print("Summary is up to date; reusing", T_SUMMARY)

display(spark.table(T_SUMMARY).select("ai_summary", "summary_error"))

Summary is up to date; reusing tedatacatalog_sandbox.te672648.cae_report_summary


ai_summary,summary_error
"This CAE report (Project No. 22-000903154, dated 20 December 2022) was prepared by TE Connectivity for customer Deutz, covering a wire distributor seal assembly identified as part B-2423482-001. The objective was to evaluate the insertion and retention forces of a cover being removed from its housing using two candidate cover materials. The analysis was performed in ANSYS 2021 R2 using solid elements (Solid187/Contact174/Target170) on a quarter-symmetry model with approximately 153,000–154,000 nodes and 98,000–98,200 elements, with a friction coefficient of 0.2 applied at all contact pairs. The two cover materials compared were Lanxess Pocan B3215 (PBT GF10, Young's modulus 3,973 MPa, tensile strength 73 MPa, elongation 3.92%) and Ultradur B4520 (PBT unfilled, Young's modulus 2,722 MPa, tensile strength 62 MPa, elongation 40.55%), while the housing was modeled with Ultradur B4300G4 (PBT GF20, Young's modulus 5,412 MPa, tensile strength 93 MPa). For insertion, Pocan B3215 produced a peak force of approximately 13.815 N per latch (55.26 N full model) and Ultradur B4520 produced approximately 10.195 N per latch (40.78 N full model), with both cover and housing assessed as safe under both materials. For retention, Pocan B3215 reached approximately 198.76 N per latch (795.04 N full model) and Ultradur B4520 reached approximately 218.16 N per latch (872.64 N full model); in both cases the cover is predicted to fail due to strains exceeding the material breaking strain, while the housing remains safe with Pocan but experiences local plastic deformation with Ultradur B4520. The conclusion is that neither material allows the cover to survive the retention load case, and a design or material change would be needed to prevent cover failure during retention.",null


## Step 6: Build the tables for Genie

Genie works best with simple tables that have typed columns and cover one subject each. This step splits the Step 4 extraction into these tables, all in `tedatacatalog_sandbox.te672648`:

| Table | One row per | Answers questions like |
| --- | --- | --- |
| `cae_report` | report | "Who is the customer? Which software was used? What is the conclusion?" |
| `cae_materials` | material | "What is the tensile strength of Ultradur B4520?" |
| `cae_load_cases` | load case or load step | "What boundary conditions were used for retention?" |
| `cae_results_summary` | process and cover material | "What is the retention force for Lanxess Pocan? Is the cover safe?" |
| `cae_result_plots` | plot and material | "What is the max strain in the housing at failure?" |
| `cae_model_data` | analysis case | "How many elements does the insertion model have?" |
| `cae_remarks` | remark | "What assumptions were made?" |

Every table has `report_number` and `path`, so they can be joined. No AI is used, so the tables are rebuilt in full on every run.

In [0]:
R = "extracted:response"
keys = extracted_df.selectExpr("path", "file_name", f"{R}:report_number::STRING AS report_number", "extracted")


def explode_list(field: str, alias: str = "x"):
    """One row per item of an array field, with a 1-based line number."""
    return (keys.selectExpr("path", "report_number",
                            f"posexplode(try_cast({R}:{field} AS ARRAY<VARIANT>)) AS (line_no, {alias})")
                .withColumn("line_no", F.col("line_no") + 1))


# cae_report: one row with the header, CAD and model info, conclusion and AI summary
report = (
    keys.selectExpr(
        "path", "file_name", "report_number",
        f"{R}:cae_team::STRING AS cae_team",
        f"{R}:project_number::STRING AS project_number",
        f"try_to_date({R}:report_date::STRING) AS report_date",
        f"{R}:part_description::STRING AS part_description",
        f"{R}:analysis_title::STRING AS analysis_title",
        f"{R}:analysis_type::STRING AS analysis_type",
        f"{R}:customer::STRING AS customer",
        f"{R}:objective::STRING AS objective",
        f"{R}:cad_part_number::STRING AS cad_part_number",
        f"{R}:cad_file_name::STRING AS cad_file_name",
        f"try_to_date({R}:cad_date::STRING) AS cad_date",
        f"{R}:software::STRING AS software",
        f"{R}:element_types::STRING AS element_types",
        f"{R}:model_simplification::STRING AS model_simplification",
        f"try_cast({R}:contact_friction AS DOUBLE) AS contact_friction",
        f"{R}:analyst::STRING AS analyst",
        f"{R}:requested_by::STRING AS requested_by",
        f"{R}:released_by::STRING AS released_by",
        f"concat_ws('; ', try_cast({R}:distribution AS ARRAY<STRING>)) AS distribution",
        f"{R}:conclusion::STRING AS conclusion",
    )
    .join(spark.table(T_SUMMARY).select("path", "ai_summary"), "path", "left")
    .join(spark.table(T_PARSED).select("path", "page_count"), "path", "left")
)
save(report, T_REPORT)

# cae_model_data
save(explode_list("model_data", "m").selectExpr(
    "report_number", "line_no", "m:case::STRING AS analysis_case",
    "try_cast(m:number_of_nodes AS BIGINT) AS number_of_nodes",
    "try_cast(m:number_of_elements AS BIGINT) AS number_of_elements", "path"), T_MODEL)

# cae_materials
save(explode_list("materials", "m").selectExpr(
    "report_number", "line_no",
    "m:designation::STRING AS material", "m:description::STRING AS material_description",
    "m:material_id::STRING AS material_id", "m:material_law::STRING AS material_law",
    "m:used_for::STRING AS used_for",
    "try_cast(m:youngs_modulus_mpa AS DOUBLE) AS youngs_modulus_mpa",
    "try_cast(m:youngs_modulus_ref_mpa AS DOUBLE) AS youngs_modulus_ref_mpa",
    "try_cast(m:tensile_strength_mpa AS DOUBLE) AS tensile_strength_mpa",
    "try_cast(m:tensile_strength_ref_mpa AS DOUBLE) AS tensile_strength_ref_mpa",
    "try_cast(m:elongation_pct AS DOUBLE) AS elongation_pct",
    "try_cast(m:elongation_ref_pct AS DOUBLE) AS elongation_ref_pct",
    "try_cast(m:poissons_ratio AS DOUBLE) AS poissons_ratio", "path"), T_MATERIALS)

# cae_load_cases
save(explode_list("load_cases", "l").selectExpr(
    "report_number", "line_no", "l:process::STRING AS process",
    "l:load_step::STRING AS load_step", "l:description::STRING AS description", "path"), T_LOAD_CASES)

# cae_results_summary
save(explode_list("results_summary", "r").selectExpr(
    "report_number", "line_no", "r:process::STRING AS process", "r:material::STRING AS cover_material",
    "try_cast(r:force_single_latch_n AS DOUBLE) AS force_single_latch_n",
    "try_cast(r:force_full_model_n AS DOUBLE) AS force_full_model_n",
    "regexp_replace(r:cover_assessment::STRING, '[.]$', '') AS cover_assessment",  # drop trailing full stop
    "regexp_replace(r:housing_assessment::STRING, '[.]$', '') AS housing_assessment",
    "r:force_text::STRING AS force_as_printed", "path"), T_RESULTS)

# cae_result_plots
save(explode_list("result_plots", "p").selectExpr(
    "report_number", "line_no", "try_cast(p:figure_number AS INT) AS figure_number",
    "p:process::STRING AS process", "p:condition::STRING AS condition",
    "p:result_type::STRING AS result_type", "p:part::STRING AS part", "p:material::STRING AS material",
    "p:unit::STRING AS unit", "try_cast(p:max_value AS DOUBLE) AS max_value",
    "try_cast(p:min_value AS DOUBLE) AS min_value", "p:observation::STRING AS observation", "path"), T_PLOTS)

# cae_remarks
save(explode_list("remarks", "r").selectExpr(
    "report_number", "line_no AS remark_no", "r::STRING AS remark", "path"), T_REMARKS)

display(spark.table(T_RESULTS))

tedatacatalog_sandbox.te672648.cae_report: 1 rows
tedatacatalog_sandbox.te672648.cae_model_data: 2 rows
tedatacatalog_sandbox.te672648.cae_materials: 3 rows
tedatacatalog_sandbox.te672648.cae_load_cases: 4 rows
tedatacatalog_sandbox.te672648.cae_results_summary: 4 rows
tedatacatalog_sandbox.te672648.cae_result_plots: 12 rows
tedatacatalog_sandbox.te672648.cae_remarks: 4 rows


report_number,line_no,process,cover_material,force_single_latch_n,force_full_model_n,cover_assessment,housing_assessment,force_as_printed,path
30044,1,Insertion,Lanxess Pocan B3215,13.815,55.26,Safe,Safe,~13.815N for single latch. ~55.26N for Full model.,dbfs:/Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf
30044,2,Insertion,Ultradur B4520,10.195,40.78,Safe,Safe,~10.195N for single latch. ~40.78N for Full model.,dbfs:/Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf
30044,3,Retention,Lanxess Pocan B3215,198.76,795.04,Failure,Safe,~198.76N for single latch. ~795.04N for Full model.,dbfs:/Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf
30044,4,Retention,Ultradur B4520,218.16,872.64,Failure,Local plastic deformation,~218.16N for single latch. ~872.64N for Full model.,dbfs:/Volumes/tedatacatalog_sandbox/te672648/josh-vol/drawings/FEM-30044_Report.pdf


## Step 7: Chart data points

`ai_parse_document` already turned the report's charts into JSON data: the insertion and retention force-displacement curves (Figures 4 and 8), and the stress-strain curve under each material table. This step reads the points directly, with no AI. The chart type comes from the caption below each chart, and the material for a stress-strain curve comes from the material table above it. The result is the `cae_curves` table, with one row per data point, so Genie can answer questions like "at what displacement does the insertion force peak?" or plot a curve.

In [0]:
w = Window.partitionBy("path").orderBy("element_index")
report_number = spark.table(T_REPORT).select("path", "report_number")

charts = (
    spark.table(T_ELEMENTS)
    .withColumn("caption", F.lead("content").over(w))        # the caption follows the chart
    .withColumn("previous_content", F.lag("content").over(w))  # a material table precedes each stress-strain chart
    .where("element_type = 'figure' AND try_parse_json(content):values IS NOT NULL")
    .selectExpr(
        "path", "page_number", "caption",
        "CASE WHEN caption ILIKE '%Insertion Force%' THEN 'insertion_force' "
        "     WHEN caption ILIKE '%Retention Force%' THEN 'retention_force' "
        "     WHEN caption ILIKE '%Material%' THEN 'stress_strain' END AS curve_type",
        "nullif(regexp_extract(previous_content, 'Material Designation</th><td>([^<]+)', 1), '') AS table_material",
        "explode(from_json(try_parse_json(content):values::STRING, 'MAP<STRING, MAP<STRING, DOUBLE>>')) AS (series, points)",
    )
    .selectExpr("*", "explode(points) AS (x_text, y)")
)

curves = (
    charts.join(report_number, "path", "left")
    .selectExpr(
        "report_number", "curve_type", "page_number",
        "CASE WHEN curve_type = 'stress_strain' THEN table_material ELSE series END AS material",
        "CASE WHEN curve_type = 'stress_strain' THEN 'true_strain' ELSE 'displacement_y_mm' END AS x_name",
        "try_cast(x_text AS DOUBLE) AS x_value",
        "CASE WHEN curve_type = 'stress_strain' THEN 'true_stress_mpa' ELSE 'force_y_n' END AS y_name",
        "y AS y_value",
        "caption", "path",
    )
    .orderBy("curve_type", "material", "x_value")
)
save(curves, T_CURVES)
display(spark.table(T_CURVES).groupBy("curve_type", "material").agg(
    F.count("*").alias("points"), F.max("y_value").alias("max_y")).orderBy("curve_type", "material"))

tedatacatalog_sandbox.te672648.cae_curves: 57 rows


curve_type,material,points,max_y
insertion_force,Lanxess Pocan,11,55.26
insertion_force,Ultradur,11,40.78
retention_force,Lanxess Pocan,7,1160.0
retention_force,Ultradur,6,872.64
stress_strain,Lanxess Pocan B3215,9,79.0
stress_strain,Ultradur B4300G4,6,94.0
stress_strain,Ultradur B4520,7,60.0


## Step 8: Read the result plots with a vision model

The strain and deformation plots in section 5 (Figures 5-7 and 9-11) are 3D ANSYS contour renders. `ai_extract` already reads their legend values, but not what the pictures show. This step sends each page image that `ai_parse_document` saved in `page_images/` to a vision model with `ai_query(..., files => content)`, with the captions of the figures on that page. A JSON schema makes the model return one row per figure and material:

* legend max and min, read from the image
* where the maximum (hottest colour) region is, e.g. latch tip, latch root, slot edge
* its colour, the view shown, and a one-sentence visual observation

The output goes to `cae_result_plot_visuals`, along with the path of the page image. The location and observation columns are AI descriptions, so treat them as qualitative; the legend values in `cae_result_plots` remain the reference. The cell checks that both sets of legend values agree. The result is cached like Steps 4 and 5.

In [0]:
PLOT_SCHEMA = {"type": "json_schema", "json_schema": {"name": "result_plots", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["plots"],
    "properties": {"plots": {"type": "array", "items": {"type": "object", "additionalProperties": False,
        "required": ["figure_number", "material", "legend_max", "legend_min", "hotspot_location",
                     "hotspot_colour", "view", "visual_observation"],
        "properties": {
            "figure_number": {"type": "integer"},
            "material": {"type": "string"},
            "legend_max": {"type": ["number", "null"]},
            "legend_min": {"type": ["number", "null"]},
            "hotspot_location": {"type": "string"},
            "hotspot_colour": {"type": "string"},
            "view": {"type": "string"},
            "visual_observation": {"type": "string"}}}}}}}}
PLOT_PROMPT = (
    "This is one page of a simulation report. It shows ANSYS contour plots, usually one plot per "
    "cover material side by side, for these figures:\n{captions}\n\nFor each figure listed and each material plot in "
    "it, return: the figure number; the material named on the plot; the Max and Min values on the colour legend; "
    "where on the part the maximum (hottest colour) region is, naming the feature (e.g. latch root, latch tip, slot "
    "edge); the colour of that region; the view shown (e.g. isometric, front); and one sentence on what the plot "
    "shows. Describe only what is visible in the image; do not use outside knowledge such as material limits."
)
VISION_EXPR = (
    f"ai_query({sql_str(VISION_ENDPOINT)}, replace({sql_str(PLOT_PROMPT)}, '{{captions}}', captions), "
    f"files => content, responseFormat => {sql_str(json.dumps(PLOT_SCHEMA))}, failOnError => false)"
)

if needs_refresh(T_PLOT_VISION, REEXTRACT):
    # Result-plot figures and their pages, from the captions ("Figure 9: von-Mises strain ... distribution ...")
    plot_figures = (
        spark.table(T_ELEMENTS)
        .where("element_type = 'caption' AND (content ILIKE '%strain%distribution%' OR content ILIKE '%deformation%')")
        .selectExpr("path", "page_number",
                    "try_cast(regexp_extract(content, '^Figure ([0-9]+):', 1) AS INT) AS figure_number",
                    "content AS caption")
        .where("figure_number IS NOT NULL")
    )
    # The page image ai_parse_document saved for each page
    page_images = (
        parsed_df
        .selectExpr("path", "parsed_at", "posexplode(try_cast(parsed:document:pages AS ARRAY<VARIANT>)) AS (page_index, pg)")
        .selectExpr("path", "parsed_at", "page_index + 1 AS page_number", "pg:image_uri::STRING AS image_path")
    )
    image_bytes = (
        spark.read.format("binaryFile").load(IMAGE_OUTPUT_PATH)
        .select(F.regexp_replace("path", "^dbfs:", "").alias("image_path"), "content")
    )
    plot_pages = (
        plot_figures.groupBy("path", "page_number")
        .agg(F.concat_ws("\n", F.sort_array(F.collect_list("caption"))).alias("captions"))
        .join(page_images, ["path", "page_number"])
        .join(image_bytes, "image_path")
    )

    plot_visuals = (
        plot_pages
        .withColumn("llm", F.expr(VISION_EXPR))
        .selectExpr("path", "parsed_at", "page_number", "image_path", "llm.errorMessage AS vision_error",
                    "posexplode_outer(try_cast(try_parse_json(llm.result):plots AS ARRAY<VARIANT>)) AS (plot_index, p)")
        .join(spark.table(T_REPORT).select("path", "report_number"), "path", "left")
        .selectExpr(
            "report_number",
            "try_cast(p:figure_number AS INT) AS figure_number",
            "p:material::STRING AS material",
            "try_cast(p:legend_max AS DOUBLE) AS legend_max",
            "try_cast(p:legend_min AS DOUBLE) AS legend_min",
            "p:hotspot_location::STRING AS hotspot_location",
            "p:hotspot_colour::STRING AS hotspot_colour",
            "p:view::STRING AS view",
            "p:visual_observation::STRING AS visual_observation",
            "page_number", "image_path", "vision_error", "path", "parsed_at",
        )
    )
    save(plot_visuals, T_PLOT_VISION)
else:
    print("Plot visuals are up to date; reusing", T_PLOT_VISION)

# Check: legend values read by the vision model vs those from ai_extract (Step 4)
display(spark.sql(f"""
    SELECT v.figure_number, v.material, v.hotspot_location,
           v.legend_max, p.max_value AS extracted_max,
           v.legend_min, p.min_value AS extracted_min,
           abs(v.legend_max - p.max_value) < 1e-6 AND abs(v.legend_min - p.min_value) < 1e-6 AS legends_agree,
           v.vision_error
    FROM {T_PLOT_VISION} v
    LEFT JOIN {T_PLOTS} p ON p.figure_number = v.figure_number AND p.material = v.material
    ORDER BY v.figure_number, v.material
"""))

Plot visuals are up to date; reusing tedatacatalog_sandbox.te672648.cae_result_plot_visuals


figure_number,material,hotspot_location,legend_max,extracted_max,legend_min,extracted_min,legends_agree,vision_error
5,Lanxess Pocan B3215,latch/slot opening area on the front face of the cover,0.041553,0.041553,1.4763E-5,1.4753E-5,true,null
5,Ultradur B4520,latch/slot opening area on the front face of the cover,0.042448,0.042448,1.1486E-5,1.1486E-5,true,null
6,Lanxess Pocan B3215,latch tip / upper latch region,0.0071178,0.0071178,1.8912E-6,1.8912E-6,true,null
6,Ultradur B4520,latch tip / upper latch region,0.012315,0.012315,1.4261E-6,1.4261E-6,true,null
7,Lanxess Pocan B3215,base/bottom body of the housing,0.0088014,0.0088014,-7.5444,-7.5444,true,null
7,Ultradur B4520,base/bottom body of the housing,0.0065001,0.0065001,-7.4577,-7.4577,true,null
9,Lanxess Pocan B3215,latch tip/root area at the top of the cover latch,0.10567,0.10567,5.4852E-7,5.4852E-7,true,null
9,Ultradur B4520,latch tip/root area at the top of the cover latch,1.0364,1.0364,8.9357E-7,8.9357E-7,true,null
10,Lanxess Pocan B3215,latch/retention arm tip region (small localised area on the inner retention feature),0.14702,0.14702,1.7685E-6,1.7685E-6,true,null
10,Ultradur B4520,latch/retention arm tip region (small localised area on the inner retention feature),0.30158,0.30158,1.8823E-6,1.8823E-6,true,null


## Step 9: Measure the colour bands of the result plots with OpenCV

The vision model in Step 8 says in words where the hotspot is. This step measures the same plots in numbers. Each ANSYS contour plot has a colour legend with 9 bands, red (max) to blue (min), and the legend's 10 values (already in the parsed text) give the value range of each band. For each of the 12 plots (6 figures x 2 cover materials), OpenCV:

1. crops the figure from the page image using the bounding box from `ai_parse_document`
2. finds each plot's legend bar (a tall, narrow coloured strip) and splits the figure at the legends; the plots are matched to materials in the order their legends appear in the parsed text
3. reads the colour of each of the 9 legend bands
4. drops the small red/green/blue axis triad, so it isn't counted as part of the model
5. assigns every coloured pixel of the model to the band with the nearest hue. Hue is used rather than full colour because the 3D shading changes how light or dark a colour is, not its hue.

The result, `cae_result_plot_colour_bands`, has one row per plot and band: the band's value range, its share of the visible coloured area, and where its pixels are (centroid as a percentage of the model's width and height in the image, 0 = left/top).

**Limits:** these are shares of the visible surface in one 2D view, not of the part's volume or of all its faces. Hidden faces are not counted, and small details depend on image resolution. Use them to compare plots, and for statements like "less than 0.2% of the visible cover is above the breaking strain", not as exact FEA results. No AI is used, so the table is rebuilt on every run.

In [0]:
import re

import cv2
import numpy as np

N_BANDS = 9          # ANSYS legends have 9 colour bands and 10 values
MAX_HUE_DIST = 8     # max hue difference (OpenCV hue scale 0-179) for a pixel to belong to a band
NUM = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"


def parse_legends(text: str):
    """[(material, [10 legend values from Max to Min])] in the order the plots appear in the parsed figure text."""
    legends = []
    for m in re.finditer(r"([^\n]+)\nType:(.*?)(?=\n[^\n]+\nType:|\Z)", text, re.S):
        values = [float(v) for v in re.findall(rf"^({NUM})(?: Max| Min)?$", m.group(2), re.M)]
        legends.append((m.group(1).strip(), values))
    return legends


def find_legend_bars(mask):
    """Legend bars are tall, narrow coloured components; returned left to right as (x, y, w, h)."""
    n, _, st, _ = cv2.connectedComponentsWithStats(mask, 8)
    return sorted(tuple(int(v) for v in st[k, :4]) for k in range(1, n) if st[k, 3] > 80 and st[k, 3] > 4 * st[k, 2])


def find_axis_triads(mask, hue):
    """Small components made of pure red, green and blue: the X/Y/Z axis triad ANSYS draws on each plot."""
    n, lab, st, _ = cv2.connectedComponentsWithStats(cv2.dilate(mask, np.ones((5, 5), np.uint8)), 8)
    triads = []
    for k in range(1, n):
        x, y, w, h, _ = st[k]
        hs = hue[(lab == k) & (mask > 0)]
        if w > 90 or h > 90 or len(hs) < 30:
            continue
        red, green, blue = np.mean((hs <= 6) | (hs >= 174)), np.mean(np.abs(hs - 60) <= 6), np.mean(np.abs(hs - 120) <= 6)
        if min(red, green, blue) > 0.08:
            triads.append((int(x), int(y), int(w), int(h)))
    return triads 


def measure_figure(fig_bgr):
    """Per plot in the figure (left to right): the colour, pixel count and centroid of each legend band."""
    hsv = cv2.cvtColor(fig_bgr, cv2.COLOR_BGR2HSV)
    hue = hsv[..., 0].astype(np.int16)
    mask = ((hsv[..., 1] > 90) & (hsv[..., 2] > 60)).astype(np.uint8)  # coloured pixels; drops white, grey, black
    for x, y, w, h in find_axis_triads(mask, hue):
        mask[y:y + h, x:x + w] = 0
    bars = find_legend_bars(mask)
    plots = []
    for j, (lx, ly, lw, lh) in enumerate(bars):
        left = max(lx - 5, 0)
        right = bars[j + 1][0] - 5 if j + 1 < len(bars) else fig_bgr.shape[1]
        # colour of each band: median of the middle of its block in the legend bar
        bh = lh / N_BANDS
        blocks = [(int(ly + i * bh + bh / 4), int(ly + (i + 1) * bh - bh / 4)) for i in range(N_BANDS)]
        mid = slice(lx + lw // 4, lx + 3 * lw // 4)
        band_hues = np.array([np.median(hue[y0:y1, mid]) for y0, y1 in blocks], dtype=np.int16)
        band_bgr = [np.median(fig_bgr[y0:y1, mid].reshape(-1, 3), axis=0) for y0, y1 in blocks]
        # coloured pixels of the model: this plot's columns, minus the legend bar
        part = mask.copy()
        part[:, :left] = 0
        part[:, right:] = 0
        part[max(ly - 3, 0): ly + lh + 3, max(lx - 3, 0): lx + lw + 3] = 0
        ys, xs = np.nonzero(part)
        dist = np.abs(hue[ys, xs][:, None] - band_hues[None, :])
        dist = np.minimum(dist, 180 - dist)  # hue is circular (red is at both 0 and 179)
        band, ok = dist.argmin(1), dist.min(1) <= MAX_HUE_DIST
        band, ys, xs = band[ok], ys[ok], xs[ok]
        x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
        bands = []
        for i in range(N_BANDS):
            sel = band == i
            bands.append(dict(
                band_index=i + 1,
                band_colour="#%02x%02x%02x" % tuple(int(c) for c in band_bgr[i][::-1]),
                pixel_count=int(sel.sum()),
                area_share_pct=float(100 * sel.mean()),
                centroid_x_pct=float(100 * (xs[sel].mean() - x0) / (x1 - x0)) if sel.any() else None,
                centroid_y_pct=float(100 * (ys[sel].mean() - y0) / (y1 - y0)) if sel.any() else None,
            ))
        plots.append(dict(plot_pixels=int(len(band)), unmatched_pixels=int((~ok).sum()), bands=bands))
    return plots


# Result-plot figures: bounding box, legend text, caption and page image (a few rows, so they are collected)
w_el = Window.partitionBy("path").orderBy("element_index")
plot_figures = (
    parsed_df
    .selectExpr("path", "try_cast(parsed:document:pages AS ARRAY<VARIANT>) AS pages",
                "posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (element_index, el)")
    .selectExpr("path", "pages", "element_index", "el:type::STRING AS element_type", "el:content::STRING AS content",
                "try_cast(el:bbox[0]:page_id AS INT) AS page_id", "try_cast(el:bbox[0]:coord AS ARRAY<INT>) AS coord")
    .withColumn("caption", F.lead("content").over(w_el))
    .where("element_type = 'figure' AND content ILIKE '%Type:%' AND caption RLIKE '^Figure [0-9]+:'")
    .selectExpr("path", "page_id + 1 AS page_number",
                "try_cast(regexp_extract(caption, '^Figure ([0-9]+):', 1) AS INT) AS figure_number", "caption",
                "content AS legend_text", "coord", "element_at(pages, page_id + 1):image_uri::STRING AS image_path")
    .join(spark.read.format("binaryFile").load(IMAGE_OUTPUT_PATH)
               .select(F.regexp_replace("path", "^dbfs:", "").alias("image_path"), "content"), "image_path")
)

band_rows = []
for fig in plot_figures.orderBy("figure_number").collect():
    page = cv2.imdecode(np.frombuffer(fig.content, np.uint8), cv2.IMREAD_COLOR)
    x1, y1, x2, y2 = fig.coord
    plots = measure_figure(page[y1:y2, x1:x2])
    legends = parse_legends(fig.legend_text)
    assert len(plots) == len(legends), f"Figure {fig.figure_number}: {len(plots)} legend bars but {len(legends)} legends in the text"
    for position, ((material, values), plot) in enumerate(zip(legends, plots), start=1):
        assert len(values) == N_BANDS + 1, f"Figure {fig.figure_number} {material}: {len(values)} legend values"
        for b in plot["bands"]:
            i = b["band_index"] - 1
            band_rows.append(dict(
                path=fig.path, figure_number=fig.figure_number, caption=fig.caption, page_number=fig.page_number,
                plot_position=position, material=material, value_from=values[i + 1], value_to=values[i],
                plot_pixels=plot["plot_pixels"], unmatched_pixels=plot["unmatched_pixels"], image_path=fig.image_path, **b))

BAND_SCHEMA = ("path STRING, figure_number INT, caption STRING, page_number INT, plot_position INT, material STRING, "
               "value_from DOUBLE, value_to DOUBLE, plot_pixels INT, unmatched_pixels INT, image_path STRING, "
               "band_index INT, band_colour STRING, pixel_count INT, area_share_pct DOUBLE, "
               "centroid_x_pct DOUBLE, centroid_y_pct DOUBLE")
colour_bands = (
    spark.createDataFrame(band_rows, BAND_SCHEMA)
    .join(spark.table(T_REPORT).select("path", "report_number"), "path", "left")
    .select("report_number", "figure_number", "material", "band_index", "band_colour", "value_from", "value_to",
            F.round("area_share_pct", 3).alias("area_share_pct"), "pixel_count",
            F.round("centroid_x_pct", 1).alias("centroid_x_pct"), F.round("centroid_y_pct", 1).alias("centroid_y_pct"),
            "plot_position", "plot_pixels", "unmatched_pixels", "caption", "page_number", "image_path", "path")
)
save(colour_bands, T_PLOT_BANDS)

# Check: one row per plot with the top band and the share of pixels no band matched (should be about 1% or less)
display(spark.sql(f"""
    SELECT figure_number, material, count(*) AS bands, round(sum(area_share_pct), 1) AS total_share_pct,
           round(100 * max(unmatched_pixels) / (max(plot_pixels) + max(unmatched_pixels)), 2) AS unmatched_pct,
           max_by(value_from, band_index = 1) AS red_band_from, max_by(value_to, band_index = 1) AS red_band_to,
           max_by(area_share_pct, band_index = 1) AS red_band_share_pct
    FROM {T_PLOT_BANDS}
    GROUP BY ALL
    ORDER BY figure_number, material
"""))

tedatacatalog_sandbox.te672648.cae_result_plot_colour_bands: 108 rows


figure_number,material,bands,total_share_pct,unmatched_pct,red_band_from,red_band_to,red_band_share_pct
5,Lanxess Pocan B3215,9,100.0,0.33,0.0392,0.041553,0.0
5,Ultradur B4520,9,100.0,0.35,0.037733,0.042448,0.0
6,Lanxess Pocan B3215,9,100.0,0.15,0.0063271,0.0071178,0.0
6,Ultradur B4520,9,100.0,0.02,0.010946,0.012315,0.0
7,Lanxess Pocan B3215,9,100.0,0.77,-0.83044,0.0088014,62.292
7,Ultradur B4520,9,100.0,0.66,-0.82285,0.0065001,62.336
9,Lanxess Pocan B3215,9,100.0,0.31,0.0392,0.10567,0.05
9,Ultradur B4520,9,100.0,0.28,0.4055,1.0364,0.073
10,Lanxess Pocan B3215,9,100.0,0.36,0.0344,0.14702,0.0
10,Ultradur B4520,9,100.0,0.27,0.0344,0.30158,0.0


## Step 10: Add table and column descriptions

Genie reads Unity Catalog comments to work out what each table and column means. Good comments, especially on units and on columns like `process` or `used_for`, give more accurate answers. This step writes a comment on each Genie table and on its important columns.

In [0]:
TABLE_COMMENTS = {
    T_REPORT: "One row per CAE report (TE CAE Report 30044): header, customer, CAD part, software, model setup, conclusion and an AI-written summary.",
    T_MATERIALS: "Material data tables of the CAE report: one row per material, with mechanical properties. used_for says whether the material is for the cover or the housing.",
    T_LOAD_CASES: "Loads and boundary conditions of the CAE report, one row per process (Insertion or Retention) and load step.",
    T_RESULTS: "Summary-of-results table of the CAE report: insertion and retention force per cover material, and whether the cover and housing are safe.",
    T_PLOTS: "Strain and deformation result plots of the CAE report: one row per figure and material, with the legend max and min values and the printed observation.",
    T_PLOT_VISION: "What each strain and deformation result plot shows, read from the report page images by a vision model: one row per figure and material, with the location of the maximum (hotspot), the view and a visual observation. Qualitative; use cae_result_plots for the legend values.",
    T_PLOT_BANDS: "OpenCV measurement of the strain and deformation result plots: one row per figure, cover material and legend colour band (1 = red/max to 9 = blue/min), with the band's value range and its share of the visible coloured model area in the 2D image. Approximate; visible surface only.",
    T_CURVES: "Data points of the charts in the CAE report: force (N) vs displacement (mm) for insertion and retention, and true stress (MPa) vs true strain for each material.",
    T_MODEL: "FE model size per analysis case (Insertion, Retention).",
    T_REMARKS: "Remarks and assumptions listed in the CAE report.",
    T_ELEMENTS: "Every element of the parsed CAE report PDF (text, tables as HTML, captions, figures with AI descriptions) with its page number. Use for full-text search.",
}
COLUMN_COMMENTS = {
    T_REPORT: {"report_number": "CAE report number, e.g. 30044", "part_description": "Part analysed",
               "customer": "Customer the analysis was done for", "contact_friction": "Contact friction coefficient used",
               "conclusion": "Conclusion as extracted from the report", "ai_summary": "AI-written summary of the report"},
    T_MATERIALS: {"material": "Material designation (trade name), e.g. Lanxess Pocan B3215",
                  "material_description": "Polymer type, e.g. PBT GF10 (PBT with 10% glass fibre)",
                  "used_for": "Cover or Housing", "youngs_modulus_mpa": "Young's modulus in MPa used in the simulation",
                  "youngs_modulus_ref_mpa": "Young's modulus in MPa printed in parentheses with * (reference value)",
                  "tensile_strength_mpa": "Tensile strength in MPa", "elongation_pct": "Elongation at break in percent (breaking strain)",
                  "poissons_ratio": "Poisson's ratio"},
    T_LOAD_CASES: {"process": "Insertion or Retention", "load_step": "Load step label (LS1, LS2, ...), null if single step"},
    T_RESULTS: {"process": "Insertion or Retention", "cover_material": "Material of the cover",
                "force_single_latch_n": "Force for one latch in newtons", "force_full_model_n": "Force for the full part (all latches) in newtons",
                "cover_assessment": "Result for the cover, e.g. Safe or Failure", "housing_assessment": "Result for the housing, e.g. Safe or Local plastic deformation"},
    T_PLOTS: {"condition": "Maximum deflection condition or Failure condition", "result_type": "Equivalent total strain or directional deformation (Y axis)",
              "part": "Cover or Housing (null for deformation plots of the assembly)", "material": "Cover material of the simulation",
              "unit": "mm/mm for strain, mm for deformation", "max_value": "Max value on the plot legend", "min_value": "Min value on the plot legend"},
    T_PLOT_VISION: {"figure_number": "Figure number in the report; joins to cae_result_plots",
                    "material": "Cover material of the simulation shown in the plot",
                    "legend_max": "Max value on the plot legend, as read by the vision model",
                    "hotspot_location": "Where on the part the maximum (hottest colour) region is, e.g. latch tip",
                    "visual_observation": "One-sentence AI description of the plot",
                    "image_path": "Volume path of the report page image that contains the plot"},
    T_PLOT_BANDS: {"figure_number": "Figure number in the report; joins to cae_result_plots",
                   "material": "Cover material of the simulation shown in the plot",
                   "band_index": "Legend band, 1 = top (red, highest values) to 9 = bottom (blue, lowest values)",
                   "band_colour": "Colour of the band as read from the legend, hex RGB",
                   "value_from": "Lower value of the band (same unit as the plot: mm/mm for strain, mm for deformation)",
                   "value_to": "Upper value of the band",
                   "area_share_pct": "Percent of the visible coloured model area in this band (2D image, approximate)",
                   "centroid_x_pct": "Horizontal position of the band's pixels, percent of the model's image width (0 = left)",
                   "centroid_y_pct": "Vertical position of the band's pixels, percent of the model's image height (0 = top)"},
    T_CURVES: {"curve_type": "insertion_force, retention_force or stress_strain",
               "material": "Material of the curve (series name for force curves)",
               "x_value": "x value: displacement along Y in mm, or true strain", "y_value": "y value: force along Y in N, or true stress in MPa"},
    T_ELEMENTS: {"element_type": "text, table, figure, caption, section_header, page_header, page_footer, ...",
                 "content": "Text of the element; tables are HTML", "figure_description": "AI description of a figure"},
}

for table, comment in TABLE_COMMENTS.items():
    spark.sql(f"COMMENT ON TABLE {table} IS {sql_str(comment)}")
    for column, col_comment in COLUMN_COMMENTS.get(table, {}).items():
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} COMMENT {sql_str(col_comment)}")
print(f"Comments written on {len(TABLE_COMMENTS)} tables")

Comments written on 11 tables


## Step 11: Build the Genie agent

**Prerequisite:** a Genie agent runs its SQL on a SQL warehouse. You need `CAN USE` on a pro or serverless SQL warehouse. At the time this notebook was written, the workspace had no warehouse you could see, so ask a workspace admin to give you one.

**Create it in the Genie UI:**
1. In the left sidebar, open **Genie** and click **New**.
2. **Add data:** from `tedatacatalog_sandbox.te672648`, select `cae_report`, `cae_materials`, `cae_load_cases`, `cae_results_summary`, `cae_result_plots`, `cae_result_plot_visuals`, `cae_result_plot_colour_bands`, `cae_curves`, `cae_model_data`, `cae_remarks` and `cae_report_elements`.
3. **Settings:** paste the title and description printed by the next cell, and choose your SQL warehouse.
4. **Instructions:** paste the text printed by the next cell.
5. **SQL examples and sample questions:** add the ones the next cell prints.
6. **Share** the agent with your team. They also need `SELECT` on the tables and `CAN USE` on the warehouse.

In [0]:
GENIE_INSTRUCTIONS = f"""These tables hold one TE Connectivity CAE (FEA) report, CAE Report 30044: a structural analysis of the cover of the Deutz-Leavy seal wire distributor, done in ANSYS.
- Start from {T_REPORT} for questions about the report itself (customer, objective, software, analyst, conclusion, summary).
- Forces are in newtons (N), displacements and deformations in mm, strain in mm/mm, stresses and moduli in MPa, elongation in percent.
- process is either Insertion (pushing the cover onto the housing) or Retention (pulling it off).
- Two cover materials are compared (see {T_MATERIALS} where used_for = 'Cover'); the housing material has used_for = 'Housing'.
- force_full_model_n is the force for the whole part; force_single_latch_n is for one latch. Use the full model unless the user asks about one latch.
- For retention, the force in {T_RESULTS} is the force at which the cover fails. The retention curve in {T_CURVES} can continue to higher forces after that point, so quote {T_RESULTS} for 'retention force'.
- Safe or fail questions: use cover_assessment and housing_assessment in {T_RESULTS}, and observation in {T_PLOTS}.
- Compare strain in {T_PLOTS} with the breaking strain elongation_pct / 100 in {T_MATERIALS} when asked whether a part breaks.
- For questions about a curve or a peak force at a displacement, use {T_CURVES}.
- For where on the part a maximum strain or deformation occurs, or what a plot looks like, use hotspot_location and visual_observation in {T_PLOT_VISION} (join to {T_PLOTS} on figure_number and material). These are AI descriptions of the images; quote the legend values from {T_PLOTS}.
- For how much of a part is in a value range (e.g. above the breaking strain), sum area_share_pct in {T_PLOT_BANDS} over the bands whose value_from is at or above the threshold. Say that it is an approximate share of the visible surface in the report image, not of the part volume.
- For anything not in the structured tables, search content and figure_description in {T_ELEMENTS} with ILIKE.
- Material names can be partial (e.g. 'Pocan', 'Ultradur'); match them with ILIKE '%name%'."""

EXAMPLE_SQLS = [
    ("What are the insertion and retention forces for each cover material?",
     f"SELECT process, cover_material, force_full_model_n, force_single_latch_n, cover_assessment, housing_assessment\nFROM {T_RESULTS}\nORDER BY process, cover_material"),
    ("Compare the mechanical properties of the materials",
     f"SELECT material, material_description, used_for, youngs_modulus_mpa, tensile_strength_mpa, elongation_pct, poissons_ratio\nFROM {T_MATERIALS}\nORDER BY used_for, material"),
    ("Which results exceed the breaking strain of the material?",
     f"SELECT p.figure_number, p.process, p.part, p.material, p.max_value AS max_strain, m.elongation_pct / 100 AS breaking_strain, p.observation\n"
     f"FROM {T_PLOTS} p\nJOIN {T_MATERIALS} m\n  ON m.material = CASE WHEN p.part = 'Housing' THEN (SELECT material FROM {T_MATERIALS} WHERE used_for = 'Housing') ELSE p.material END\n"
     f"WHERE p.unit = 'mm/mm' AND p.max_value > m.elongation_pct / 100\nORDER BY p.figure_number"),
    ("Show the insertion force curve",
     f"SELECT material, x_value AS displacement_mm, y_value AS force_n\nFROM {T_CURVES}\nWHERE curve_type = 'insertion_force'\nORDER BY material, x_value"),
]
SAMPLE_QUESTIONS = [
    "What is this CAE report about?",
    "Which cover material needs the lower insertion force?",
    "Is the cover safe during retention?",
    "What boundary conditions were used for the retention analysis?",
    "Which material has the highest tensile strength?",
    "Where is the maximum strain in the cover at failure?",
    "How much of the visible cover is above the breaking strain at failure?",
]

print(f"TITLE\n{GENIE_TITLE}\n\nDESCRIPTION\n{GENIE_DESCRIPTION}\n")
print("INSTRUCTIONS\n" + GENIE_INSTRUCTIONS + "\n")
for q, sql in EXAMPLE_SQLS:
    print(f"SQL EXAMPLE: {q}\n{sql}\n")
print("SAMPLE QUESTIONS\n" + "\n".join(SAMPLE_QUESTIONS))

# Check that every example SQL runs and returns rows
for q, sql in EXAMPLE_SQLS:
    print(f"{spark.sql(sql).count():>4} rows  {q}")

TITLE
CAE Report 30044 Assistant

DESCRIPTION
Ask questions about TE CAE report 30044 (structural analysis of the Deutz-Leavy seal wire distributor cover): materials, load cases, insertion and retention forces, strain and deformation results.

INSTRUCTIONS
These tables hold one TE Connectivity CAE (FEA) report, CAE Report 30044: a structural analysis of the cover of the Deutz-Leavy seal wire distributor, done in ANSYS.
- Start from tedatacatalog_sandbox.te672648.cae_report for questions about the report itself (customer, objective, software, analyst, conclusion, summary).
- Forces are in newtons (N), displacements and deformations in mm, strain in mm/mm, stresses and moduli in MPa, elongation in percent.
- process is either Insertion (pushing the cover onto the housing) or Retention (pulling it off).
- Two cover materials are compared (see tedatacatalog_sandbox.te672648.cae_materials where used_for = 'Cover'); the housing material has used_for = 'Housing'.
- force_full_model_n is the f

In [0]:
# Run check: row counts per table, and the key results side by side with the chart data they should match
display(spark.createDataFrame([(t.split(".")[-1], spark.table(t).count()) for t in GENIE_TABLES],
                              "table STRING, row_count LONG"))

# Each full-model force in the results table should be a point on the matching force curve.
# (For retention it is the force at failure, which can be below the curve's peak.)
display(spark.sql(f"""
    SELECT r.process, r.cover_material, r.force_full_model_n,
           max(c.y_value) AS curve_peak_force_n,
           min(CASE WHEN abs(c.y_value - r.force_full_model_n) < 0.01 THEN c.x_value END) AS displacement_at_reported_force_mm,
           count_if(abs(c.y_value - r.force_full_model_n) < 0.01) > 0 AS reported_force_on_curve
    FROM {T_RESULTS} r
    LEFT JOIN {T_CURVES} c
      ON c.curve_type = (lower(r.process) || '_force')
     AND r.cover_material ILIKE ('%' || c.material || '%')
    GROUP BY ALL
    ORDER BY r.process, r.cover_material
"""))

# Load cases: Insertion should have 1 load, Retention the load steps LS1-LS3
display(spark.table(T_LOAD_CASES).select("process", "load_step", "description"))

table,row_count
cae_report,1
cae_materials,3
cae_load_cases,4
cae_results_summary,4
cae_result_plots,12
cae_result_plot_visuals,12
cae_result_plot_colour_bands,108
cae_curves,57
cae_model_data,2
cae_remarks,4


process,cover_material,force_full_model_n,curve_peak_force_n,displacement_at_reported_force_mm,reported_force_on_curve
Insertion,Lanxess Pocan B3215,55.26,55.26,6.5,true
Insertion,Ultradur B4520,40.78,40.78,6.5,true
Retention,Lanxess Pocan B3215,795.04,1160.0,0.62,true
Retention,Ultradur B4520,872.64,872.64,2.36,true


process,load_step,description
Insertion,null,Remote displacement of 9 mm is applied on rigid body along -ve Y direction and fixed in all other direction.
Retention,LS1,Latch is deflected 2 mm in -ve X direction and fixed in all other directions.
Retention,LS2,Applied defelection is released.
Retention,LS3,Remote displacement of 9 mm is applied on rigid body along +ve Y direction and fixed in all other direction.
